<a href="https://colab.research.google.com/github/impo33ibru/dap-2024/blob/main/les06/lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа №6. Линейная регрессия

Выполните следующие задания:

1. Откройте в файл в Google Colab (используйте собственный форк репозитория).
2. Решите задачи.
3. Сохраните результат в ваш репозиторий github в директорию ./les06
4. Создайте pull request в репозиторий https://github.com/chebotarevsa/dap-2024. Название pull request должно иметь формат "<Номер лабораторной работы>  <Номер группы> <ФИО>"
5. Сдайте работу в системе "Пегас", в отчет укажите ссылку на pull request

Набор данных Diabetes (Диабет) содержит 442 образца с 10-ю признаками: возраст, пол, индекс массы тела, средний показатель давления крови и шесть измерений сыворотки крови. Целевое значение - количественный показатель прогрессирования заболевания через год после анализов.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Извлечение данных

In [ ]:
from sklearn import datasets
diabetes = datasets.load_diabetes()

1. Выведете описание набора данных и наименование признаков.

In [ ]:
print(diabetes.DESCR)
print(diabetes.feature_names)

In [ ]:
.. _diabetes_dataset:

Diabetes dataset
----------------

Ten baseline variables, age, sex, body mass index, average blood
pressure, and six blood serum measurements were obtained for each of n =
442 diabetes patients, as well as the response of interest, a
quantitative measure of disease progression one year after baseline.

**Data Set Characteristics:**

:Number of Instances: 442

:Number of Attributes: First 10 columns are numeric predictive values

:Target: Column 11 is a quantitative measure of disease progression one year after baseline

:Attribute Information:
    - age     age in years
    - sex
    - bmi     body mass index
    - bp      average blood pressure
    - s1      tc, total serum cholesterol
    - s2      ldl, low-density lipoproteins
...
Bradley Efron, Trevor Hastie, Iain Johnstone and Robert Tibshirani (2004) "Least Angle Regression," Annals of Statistics (with discussion), 407-499.
(https://web.stanford.edu/~hastie/Papers/LARS/LeastAngle_2002.pdf)

['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...

### Подготовка данных

2. Из загруженного набора данных создайте DataFrame, содержащий как признаки, так и целевое значение. Выведите первые 5 строк набора.

In [ ]:
diabetes_df = pd.DataFrame(data = diabetes.data, columns = diabetes.feature_names)
diabetes_df['target'] = diabetes.target
diabetes_df.head(5)

<div id="df-7820afbc-9e25-4a92-87d3-ce8607cf40d5" class="colab-df-container">
    <div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>age</th>
      <th>sex</th>
      <th>bmi</th>
      <th>bp</th>
      <th>s1</th>
      <th>s2</th>
      <th>s3</th>
      <th>s4</th>
      <th>s5</th>
      <th>s6</th>
      <th>target</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>0.038076</td>
      <td>0.050680</td>
      <td>0.061696</td>
      <td>0.021872</td>
      <td>-0.044223</td>
      <td>-0.034821</td>
      <td>-0.043401</td>
      <td>-0.002592</td>
      <td>0.019907</td>
      <td>-0.017646</td>
      <td>151.0</td>
    </tr>
    <tr>
      <th>1</th>
      <td>-0.001882</td>
      <td>-0.044642</td>
      <td>-0.051474</td>
      <td>-0.026328</td>
      <td>-0.008449</td>
      <td>-0.019163</td>
      <td>0.074412</td>
      <td>-0.039493</td>
      <td>-0.068332</td>
      <td>-0.092204</td>
      <td>75.0</td>
    </tr>
    <tr>
      <th>2</th>
      <td>0.085299</td>
      <td>0.050680</td>
      <td>0.044451</td>
      <td>-0.005670</td>
      <td>-0.045599</td>
      <td>-0.034194</td>
      <td>-0.032356</td>
      <td>-0.002592</td>
      <td>0.002861</td>
      <td>-0.025930</td>
      <td>141.0</td>
    </tr>
    <tr>
      <th>3</th>
      <td>-0.089063</td>
      <td>-0.044642</td>
      <td>-0.011595</td>
      <td>-0.036656</td>
      <td>0.012191</td>
      <td>0.024991</td>
      <td>-0.036038</td>
      <td>0.034309</td>
      <td>0.022688</td>
      <td>-0.009362</td>
      <td>206.0</td>
    </tr>
    <tr>
      <th>4</th>
      <td>0.005383</td>
      <td>-0.044642</td>
      <td>-0.036385</td>
      <td>0.021872</td>
      <td>0.003935</td>
      <td>0.015596</td>
      <td>0.008142</td>
      <td>-0.002592</td>
      <td>-0.031988</td>
      <td>-0.046641</td>
      <td>135.0</td>
    </tr>
  </tbody>
</table>
  </div>

3. Выведете информацию о типах данных в наборе. Имеются ли в наборе категориальные признаки? Имеются ли в наборе данные имеющие значение null?

In [ ]:
print(diabetes_df.info())
print("\nЗдесь нет категориальных признаков, поскольку все признаки являются числами.\n")
print(diabetes_df.isnull().sum())
print("\nВо фрейме данных нет нулевых значений.\n")

In [ ]:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   age     442 non-null    float64
 1   sex     442 non-null    float64
 2   bmi     442 non-null    float64
 3   bp      442 non-null    float64
 4   s1      442 non-null    float64
 5   s2      442 non-null    float64
 6   s3      442 non-null    float64
 7   s4      442 non-null    float64
 8   s5      442 non-null    float64
 9   s6      442 non-null    float64
 10  target  442 non-null    float64
dtypes: float64(11)
memory usage: 38.1 KB
None

Здесь нет категориальных признаков, поскольку все признаки являются числами.

age       0
sex       0
bmi       0
...
dtype: int64

Во фрейме данных нет нулевых значений.

Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...

## Исследование данных

4. Постройте матрицу корреляции.

In [ ]:
matr = diabetes_df.corr()
matr

<div id="df-da9e8303-6dff-408f-8582-e77549d35af8" class="colab-df-container">
    <div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>age</th>
      <th>sex</th>
      <th>bmi</th>
      <th>bp</th>
      <th>s1</th>
      <th>s2</th>
      <th>s3</th>
      <th>s4</th>
      <th>s5</th>
      <th>s6</th>
      <th>target</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>age</th>
      <td>1.000000</td>
      <td>0.173737</td>
      <td>0.185085</td>
      <td>0.335428</td>
      <td>0.260061</td>
      <td>0.219243</td>
      <td>-0.075181</td>
      <td>0.203841</td>
      <td>0.270774</td>
      <td>0.301731</td>
      <td>0.187889</td>
    </tr>
    <tr>
      <th>sex</th>
      <td>0.173737</td>
      <td>1.000000</td>
      <td>0.088161</td>
      <td>0.241010</td>
      <td>0.035277</td>
      <td>0.142637</td>
      <td>-0.379090</td>
      <td>0.332115</td>
      <td>0.149916</td>
      <td>0.208133</td>
      <td>0.043062</td>
    </tr>
    <tr>
      <th>bmi</th>
      <td>0.185085</td>
      <td>0.088161</td>
      <td>1.000000</td>
      <td>0.395411</td>
      <td>0.249777</td>
      <td>0.261170</td>
      <td>-0.366811</td>
      <td>0.413807</td>
      <td>0.446157</td>
      <td>0.388680</td>
      <td>0.586450</td>
    </tr>
    <tr>
      <th>bp</th>
      <td>0.335428</td>
      <td>0.241010</td>
      <td>0.395411</td>
      <td>1.000000</td>
      <td>0.242464</td>
      <td>0.185548</td>
      <td>-0.178762</td>
      <td>0.257650</td>
      <td>0.393480</td>
      <td>0.390430</td>
      <td>0.441482</td>
    </tr>
    <tr>
      <th>s1</th>
      <td>0.260061</td>
      <td>0.035277</td>
      <td>0.249777</td>
      <td>0.242464</td>
      <td>1.000000</td>
      <td>0.896663</td>
      <td>0.051519</td>
      <td>0.542207</td>
      <td>0.515503</td>
      <td>0.325717</td>
      <td>0.212022</td>
    </tr>
    <tr>
      <th>s2</th>
      <td>0.219243</td>
      <td>0.142637</td>
      <td>0.261170</td>
      <td>0.185548</td>
      <td>0.896663</td>
      <td>1.000000</td>
      <td>-0.196455</td>
      <td>0.659817</td>
      <td>0.318357</td>
      <td>0.290600</td>
      <td>0.174054</td>
    </tr>
    <tr>
      <th>s3</th>
      <td>-0.075181</td>
      <td>-0.379090</td>
      <td>-0.366811</td>
      <td>-0.178762</td>
      <td>0.051519</td>
      <td>-0.196455</td>
      <td>1.000000</td>
      <td>-0.738493</td>
      <td>-0.398577</td>
      <td>-0.273697</td>
      <td>-0.394789</td>
    </tr>
    <tr>
      <th>s4</th>
      <td>0.203841</td>
      <td>0.332115</td>
      <td>0.413807</td>
      <td>0.257650</td>
      <td>0.542207</td>
      <td>0.659817</td>
      <td>-0.738493</td>
      <td>1.000000</td>
      <td>0.617859</td>
      <td>0.417212</td>
      <td>0.430453</td>
    </tr>
    <tr>
      <th>s5</th>
      <td>0.270774</td>
      <td>0.149916</td>
      <td>0.446157</td>
      <td>0.393480</td>
      <td>0.515503</td>
      <td>0.318357</td>
      <td>-0.398577</td>
      <td>0.617859</td>
      <td>1.000000</td>
      <td>0.464669</td>
      <td>0.565883</td>
    </tr>
    <tr>
      <th>s6</th>
      <td>0.301731</td>
      <td>0.208133</td>
      <td>0.388680</td>
      <td>0.390430</td>
      <td>0.325717</td>
      <td>0.290600</td>
      <td>-0.273697</td>
      <td>0.417212</td>
      <td>0.464669</td>
      <td>1.000000</td>
      <td>0.382483</td>
    </tr>
    <tr>
      <th>target</th>
      <td>0.187889</td>
      <td>0.043062</td>
      <td>0.586450</td>
      <td>0.441482</td>
      <td>0.212022</td>
      <td>0.174054</td>
      <td>-0.394789</td>
      <td>0.430453</td>
      <td>0.565883</td>
      <td>0.382483</td>
      <td>1.000000</td>
    </tr>
  </tbody>
</table>
</div>
    

  </div>

5. Постройте диаграмму рассеяния целевого значение и признака, коэффициент корреляции которого с  целевым значением, самый высокий.

In [ ]:
feature = matr.loc[matr['target'] != 1, 'target'].idxmax()
x = diabetes_df[feature]
y = diabetes_df['target']
plt.scatter(x,y)
plt.xlabel(feature)
plt.ylabel("target")
plt.title("Диаграмма рассеивания")

In [ ]:
Text(0.5, 1.0, 'Диаграмма рассеивания')

In [ ]:
<Figure size 640x480 with 1 Axes>

6. Сформируйте набор признаков (X) из 5 признаков с самым высоким коэффициентом корреляции с целевым значением. Сформируйте набор для целевого значения (y).

In [ ]:
features = matr.loc[matr['target'] != 1, 'target'].abs().nlargest(5).index
x = diabetes_df[features]
y = diabetes_df['target']
print(x, y)

In [ ]:
          bmi        s5        bp        s4        s3
0    0.061696  0.019907  0.021872 -0.002592 -0.043401
1   -0.051474 -0.068332 -0.026328 -0.039493  0.074412
2    0.044451  0.002861 -0.005670 -0.002592 -0.032356
3   -0.011595  0.022688 -0.036656  0.034309 -0.036038
4   -0.036385 -0.031988  0.021872 -0.002592  0.008142
..        ...       ...       ...       ...       ...
437  0.019662  0.031193  0.059744 -0.002592 -0.028674
438 -0.015906 -0.018114 -0.067642  0.034309 -0.028674
439 -0.015906 -0.046883  0.017293 -0.011080 -0.024993
440  0.039062  0.044529  0.001215  0.026560 -0.028674
441 -0.073030 -0.004222 -0.081413 -0.039493  0.173816

[442 rows x 5 columns] 0      151.0
1       75.0
2      141.0
3      206.0
4      135.0
       ...
437    178.0
438    104.0
439    132.0
440    220.0
441     57.0
Name: target, Length: 442, dtype: float64

## Предсказательная модель

7. Разделите набор данных на два, одни для обучения модели другой для проверки. Тестовый набор должен содержать 25 процентов данных.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x, y, train_size=0.25, random_state=42)

8. Выполните обучение модели.

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
print(y_pred[:5])
print(y_test[:5])

In [ ]:
[132.80992445 175.05566388 151.18309658 234.55669373 120.24582926]
287    219.0
211     70.0
72     202.0
321    230.0
73     111.0
Name: target, dtype: float64

## Проверка модели

9. Расчитайте Root mean squared error (RMSE)

In [ ]:
from sklearn import metrics
print('Среднеквадратичная ошибка (RMSE) = ', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

In [ ]:
Среднеквадратичная ошибка (RMSE) =  58.51740342454687

10. Расчитайте R² (коэффициент детерминации)

In [ ]:
print('R2 = ', np.sqrt(metrics.r2_score(y_test, y_pred)))

In [ ]:
R2 =  0.6465342225766915

## Вопросы для защиты

1. Какие типы машинного обучения вы знаете?
2. Чем отличается обучение с учителем и без учителя?
3. Чем пакетное обучение отличается от динамического?
4. Чем обучение на основе образцов отличается от обучения на основе модели?
5. Что такое линейная регрессия?
6. Что такое градиентный спуск?
7. Как правильно обрабатывать категориальные признаки?
8. Что такое матрица корреляции?
9. Что показывает метрика RMSE?
10. Что показывает метрика R²?